# World Model Demo Notebook

This notebook shows how to:
1. Create a LIBERO task environment
2. Connect to the running world-model server
3. Roll out actions in imagination
4. Export videos for comparison

**Prerequisites**
- Start the world-model server first (`bash start_server.sh`)
- Ensure LIBERO datasets are available locally

In [ ]:
from libero.libero import benchmark
from libero.libero import get_libero_path
from libero.libero.envs import OffScreenRenderEnv

import pathlib
import numpy as np
import math
import os

CAMERA_NAMES = ["agentview", "birdview", "robot0_eye_in_hand", "sideview", "canonical_frontview"]

def _get_libero_env(task, resolution, seed):
    """Initializes and returns the LIBERO environment, along with the task description."""
    task_description = task.language
    task_bddl_file = pathlib.Path(get_libero_path("bddl_files")) / task.problem_folder / task.bddl_file
    env_args = {"bddl_file_name": task_bddl_file, "camera_heights": resolution, "camera_widths": resolution, "camera_names": CAMERA_NAMES}
    env = OffScreenRenderEnv(**env_args)
    env.seed(seed)  # IMPORTANT: seed seems to affect object positions even when using fixed initial state
    return env, task_description

def _get_empty_env(task, env):
    import robosuite
    dataset_file = os.path.join(get_libero_path("datasets"), f"{task.problem_folder}/{task.name}_demo.hdf5")
    import h5py
    import json
    f = h5py.File(dataset_file, "r")
    env_meta = json.loads(f["data"].attrs["env_args"])
    f.close()
    empty_env_kwargs = env_meta['env_kwargs'].copy()
    empty_env_kwargs['env_name'] = "SingleArmEmptyEnv"
    empty_env_kwargs['hard_reset'] = False
    empty_env_kwargs['ignore_done'] = True
    empty_env_kwargs['has_offscreen_renderer'] = False
    empty_env_kwargs['has_renderer'] = False
    empty_env_kwargs['use_camera_obs'] = False
    empty_env_kwargs['camera_names'] = CAMERA_NAMES
    empty_env_kwargs['camera_heights'] = LIBERO_ENV_RESOLUTION
    empty_env_kwargs['camera_widths'] = LIBERO_ENV_RESOLUTION
    empty_env_kwargs['robots'] = [type(robot.robot_model).__name__ for robot in env.robots]
    empty_env = robosuite.make(**empty_env_kwargs)
    empty_env.copy_env_model(env) # only copy the camera model and robot initial position, Line 108 environments/robosuite/robosuite/environments/manipulation/empty_env.py
    return empty_env


task_suite_name = "libero_10"
task_id = 6
seed = 0

LIBERO_ENV_RESOLUTION = 224
LIBERO_DUMMY_ACTION = [0.0] * 6 + [-1.0]

benchmark_dict = benchmark.get_benchmark_dict()
task_suite = benchmark_dict[task_suite_name]()
task = task_suite.get_task(task_id)
dataset_path = os.path.join(get_libero_path("datasets"), task_suite.get_task_demonstration(task_id))
initial_states = task_suite.get_task_init_states(task_id)
env, task_description = _get_libero_env(task, LIBERO_ENV_RESOLUTION, seed)
empty_env = _get_empty_env(task, env)
print(f"Task description: {task_description}")

## Connect to the world-model server

Update `host` / `port` if your server is running elsewhere.

In [ ]:
from wm_client.client import WMClient
from wm_client.wm_env import WMEnv
host = "0.0.0.0"
port = 7880
wm_client = WMClient(host, port)

## Initialize wrapped environment

`WMEnv` combines the simulator environment with the world-model client so each step can collect conditioning frames and predictions.

In [ ]:
env.reset()
# wm_env = env
wm_env = WMEnv(env, empty_env, wm_client, panel_cams=["agentview", "birdview", "robot0_eye_in_hand", "sideview"])
num_steps = 0
done = False
replay_images = []
obs = wm_env.reset()

## Load a demonstration trajectory

This loads `demo_0` actions/states from the LIBERO dataset and resets the simulator to the first demonstration state.

In [ ]:
import h5py
f = h5py.File(dataset_path, "r")
actions = f["data"]["demo_0"]["actions"][:]
states = f["data"]["demo_0"]["states"][:]
f.close()
obs = env.set_init_state(states[0])


## Run environment rollout from demonstration actions

This executes real environment steps and records agent-view frames into `replay_images`.

In [ ]:
for action in actions[:110]:
    obs, reward, done, info = wm_env.step(action)
    replay_images.append(obs['agentview_image'][::-1])
for _ in range(10):
    obs, reward, done, info = wm_env.step(LIBERO_DUMMY_ACTION)
    replay_images.append(obs['agentview_image'][::-1])

## Imagine rollout with user-defined action sequence

Provide a list of 7D actions `[dx, dy, dz, droll, dpitch, dyaw, gripper]`. The output video is saved as `imagined_rollout.mp4`.

In [ ]:
actions = [[0,0.5,-0.3,0,0,0,-1]] * 40
# actions = [[-0.5,0,-0.3,0,0,0,-1]] * 40
with wm_env.simulation():
    imagine_output = wm_env.simulate(actions)
    imagine_obs_chunk = imagine_output['future_obs']
    imagine_obs = imagine_obs_chunk[-1]
    # next_action = policy(imagine_obs) # you can use the obs just like environment observations, where images are generated from world model, and proprio states are computed through robot dynamics of the actions
import imageio
imageio.mimwrite("imagined_rollout.mp4", imagine_output['WMPredictionOutput'].full_video, fps=20)

## Compare with real rollout video

This replays the same action list in the simulator and saves `environment_rollout.mp4`.

In [ ]:
for action in actions:
    obs, reward, done, info = wm_env.step(action)
    replay_images.append(obs['agentview_image'][::-1])
imageio.mimwrite("environment_rollout.mp4", list(wm_env.history_frames)[-40:], fps=20)

## Export replay video

Writes collected frames to `test_replay.mp4` and clears the replay buffer.

In [ ]:
import imageio
imageio.mimwrite("replay.mp4", replay_images, fps=20)
replay_images = []